# 06 — Full fine-tuning (TRL `SFTTrainer`, no peft)

Full fine-tuning (FFT) updates **all** model weights — strictly more expressive than LoRA,
and much more memory-hungry. The code is the same TRL `SFTTrainer` as notebook 05 **minus
the `peft_config`**.

**Library focus:** `trl.SFTTrainer` / `SFTConfig` without peft.

**Memory (single 96 GB GPU):** a 4B model in bf16 needs weights + grads + Adam states
(~8B params of fp32 optimizer state) ≈ 50–70 GB. Keep it in budget with
`gradient_checkpointing=True`, a small per-device batch with gradient accumulation, and
optionally an 8-bit optimizer (`optim="adamw_bnb_8bit"`). Larger models need LoRA (05).

In [ ]:
import pathlib

MODEL = "Qwen/Qwen3.5-4B"
LIMIT = 256
OUT = pathlib.Path("/workspaces/llm-research/runs/_examples_06_fft")

In [ ]:
from datasets import load_dataset

ds = load_dataset("MedRAG/textbooks", split="train").select(range(LIMIT))
ds = ds.rename_column("content", "text").select_columns(["text"])
print(ds)

## Configure & train

Note the **lower learning rate** than LoRA (full weights are sensitive: ~1e-5–2e-5 vs
2e-4), `gradient_checkpointing=True`, and a small batch. No `peft_config` is passed, so every
parameter trains.

In [ ]:
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

cfg = SFTConfig(
    output_dir=str(OUT),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=1e-5,  # full-weight FT: ~10x lower than LoRA
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,  # key memory lever for FFT
    optim="adamw_torch",  # "adamw_bnb_8bit" to halve optimizer-state memory
    logging_steps=10,
    report_to="none",  # "wandb" to log curves
    seed=0,
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=MODEL,
    args=cfg,
    train_dataset=ds,
    processing_class=tok,
    # no peft_config -> full fine-tuning
)
print("all params train:", f"{trainer.model.num_parameters() / 1e9:.2f}B")

In [ ]:
result = trainer.train()
model_dir = OUT / "model"
trainer.save_model(str(model_dir))  # standalone full checkpoint (large)
print("train_loss:", result.training_loss, "| checkpoint ->", model_dir)

The saved checkpoint is a standalone model — load it like any other with
`AutoModelForCausalLM.from_pretrained(model_dir)`, serve it with vLLM, or evaluate it
(notebook 07). (For a *VLM* base, a text-only full checkpoint may not register with vLLM —
evaluate with the HF backend in that case.)

> **Project pointer:** the repo currently wraps only LoRA (`llm_core.training.sft.train_lora`);
> full fine-tuning is this raw TRL path. The same `mix_corpora` data prep from
> `scripts/train.py` applies.